# RelCheck v8 — Knowledge Base Architecture (10 images)

## Architecture Overview

**Key shift from v3–v7:** Instead of verifying triples one-at-a-time with yes/no questions,
we first build a **Visual Knowledge Base (KB)** of verified facts about the image,
then compare the caption against it. Inspired by Woodpecker (2024), extended to relations.

**4 Stages:**
1. **Caption** — BLIP-2 (multi-prompt) → Llama merge
2. **Visual KB** — GroundingDINO (objects, counts, bboxes) + bbox geometry (spatial facts) + Maverick VLM (action/attribute descriptions, missing objects)
3. **Compare** — Llama extracts triples from caption, checks each against KB → SUPPORTED / CONTRADICTED / UNVERIFIABLE
4. **Correct** — Llama edits ONLY contradictions using KB evidence as source of truth

**Model roles (no same-model bias):**
| Model | Role | Sees image? |
|-------|------|------------|
| BLIP-2 | Generate caption (TARGET) | Yes |
| GroundingDINO | Object detection — hard evidence | Yes (deterministic) |
| Maverick VLM | Describe relationships between detected objects | Yes |
| Llama-3.3-70B | Compare + correct (text only) | **No** |

**Evaluation:** Mini R-POPE (LLM-judge) using R-Bench ground truth.


In [ ]:
# ============================================================
# Cell 1 — Install + Setup
# ============================================================
!pip install -q together transformers pillow torch torchvision
!pip install -q spacy
!python -m spacy download en_core_web_sm -q

import os, json, base64, re, time, textwrap, glob, random
from io import BytesIO
from collections import defaultdict, Counter
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from together import Together
from transformers import (
    AutoProcessor, AutoModelForZeroShotObjectDetection,
)

# ── API Key ──
TOGETHER_API_KEY = ""  # <-- paste your key here
os.environ["TOGETHER_API_KEY"] = TOGETHER_API_KEY
client = Together(api_key=TOGETHER_API_KEY)

# ── Model IDs ──
CAPTION_MODEL = "Qwen/Qwen3-VL-8B-Instruct"       # Captioner (strong on objects)
VLM_MODEL = "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"  # Verifier (cross-model)
LLM_MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"            # Compare + Correct
GDINO_ID  = "IDEA-Research/grounding-dino-tiny"                    # Object detection

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device:    {DEVICE}")
print(f"Captioner: {CAPTION_MODEL}")
print(f"Verifier:  {VLM_MODEL}")
print(f"LLM:       {LLM_MODEL}")
print(f"GDino:     {GDINO_ID}")


In [ ]:
# ============================================================
# Cell 2 — Load Models (GroundingDINO only — captioning via API)
# ============================================================
# BLIP-2 removed — using Qwen3-VL-8B via Together.ai for captioning.
# This frees GPU memory for GroundingDINO.

print("Loading GroundingDINO...")
gdino_processor = AutoProcessor.from_pretrained(GDINO_ID)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_ID).to(DEVICE)
print("  GroundingDINO loaded.")

print(f"\nReady on {DEVICE}. Captioning via Together.ai ({CAPTION_MODEL}).")


In [ ]:
# ============================================================
# Cell 3 — Load Images + Download R-Bench Data
# ============================================================
from google.colab import drive
from pathlib import Path

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

DRIVE_IMAGES_DIR = "/content/drive/MyDrive/RelCheck_Data/images"
RBENCH_PATH = "/content/drive/MyDrive/RelCheck_Data/rbench_data.json"

if not os.path.isdir(DRIVE_IMAGES_DIR):
    raise FileNotFoundError(f"No images at {DRIVE_IMAGES_DIR}. Run Master notebook first.")

# ── Load test images ──
cached = list(Path(DRIVE_IMAGES_DIR).glob("*.jpg")) + list(Path(DRIVE_IMAGES_DIR).glob("*.jpeg"))
print(f"Found {len(cached)} cached images.")

N_IMAGES = 10
random.seed(42)
selected = random.sample(cached, min(N_IMAGES, len(cached)))

images = {}
for p in selected:
    images[p.stem] = Image.open(str(p)).convert("RGB")

print(f"Loaded {len(images)} images:")
for img_id in images:
    print(f"  {img_id}: {images[img_id].size}")

# ── Download R-Bench annotations (if not already on Drive) ──
rbench_data = None

if os.path.exists(RBENCH_PATH):
    with open(RBENCH_PATH) as f:
        rbench_data = json.load(f)
    print(f"\nR-Bench loaded from Drive: {len(rbench_data)} entries.")
else:
    print("\nR-Bench not on Drive — downloading annotations (~5 MB)...")
    try:
        !pip install -q gdown
        import zipfile, shutil

        GDRIVE_FILE_ID = "1sqO0MWBg_HXp5cIKb-nstjNEEk5crUWH"
        ZIP_PATH = "/tmp/rbench_annotations.zip"
        EXTRACT_DIR = "/tmp/rbench_extracted"

        !gdown --id {GDRIVE_FILE_ID} -O {ZIP_PATH} -q

        if os.path.exists(ZIP_PATH):
            os.makedirs(EXTRACT_DIR, exist_ok=True)
            with zipfile.ZipFile(ZIP_PATH, 'r') as z:
                z.extractall(EXTRACT_DIR)

            # Find the annotation JSON (image-level_filterd.json)
            import pathlib
            all_jsons = sorted(pathlib.Path(EXTRACT_DIR).rglob("*.json"))
            print(f"  Found {len(all_jsons)} JSON files in archive")

            # Try to find image-level annotations
            annot_file = None
            for f in all_jsons:
                try:
                    with open(f) as fh:
                        data = json.load(fh)
                    if isinstance(data, list) and len(data) > 50:
                        sample = str(data[0]).lower()
                        if any(k in sample for k in ['question', 'text', 'label', 'answer']):
                            annot_file = f
                            rbench_data = data
                            break
                except:
                    continue

            if rbench_data is not None:
                # Normalize entries
                normalized = []
                for entry in rbench_data:
                    norm = {}
                    # Image ID — try multiple key names
                    for k in ['image', 'image_id', 'img_id', 'img']:
                        if k in entry:
                            norm['image'] = str(entry[k])
                            break
                    # Question
                    for k in ['text', 'question', 'sent']:
                        if k in entry:
                            norm['question'] = str(entry[k])
                            break
                    # Answer
                    for k in ['label', 'answer']:
                        if k in entry:
                            norm['answer'] = str(entry[k]).strip().lower()
                            break
                    if 'image' in norm and 'question' in norm and 'answer' in norm:
                        if norm['answer'] in ('yes', 'no'):
                            normalized.append(norm)

                rbench_data = normalized

                # Save to Drive for future use
                with open(RBENCH_PATH, 'w') as f:
                    json.dump(rbench_data, f)
                print(f"  ✅ R-Bench downloaded and saved: {len(rbench_data)} questions")
                print(f"  Saved to {RBENCH_PATH}")

                # Show sample
                print(f"  Sample entry: {rbench_data[0]}")
                unique_imgs = set(e['image'] for e in rbench_data)
                print(f"  Unique images: {len(unique_imgs)}")
            else:
                print("  ⚠️ Could not find annotation file in archive")
    except Exception as e:
        print(f"  ⚠️ R-Bench download failed: {e}")
        print("  R-POPE evaluation will be skipped.")

# ── Match R-Bench images to our test images ──
if rbench_data:
    # Build lookup: R-Bench image field → list of questions
    rbench_by_image = defaultdict(list)
    for entry in rbench_data:
        rbench_by_image[entry['image']].append(entry)

    # Try to match our image filenames to R-Bench image fields
    matched = 0
    for our_id in images:
        # Try exact match, then with extensions, then substring
        for rb_key in rbench_by_image:
            rb_clean = rb_key.replace('.jpg', '').replace('.jpeg', '').replace('.png', '')
            if our_id == rb_clean or our_id in rb_key or rb_key in our_id:
                matched += 1
                break

    print(f"\nR-Bench coverage: {matched}/{len(images)} test images have questions")
    if matched == 0:
        print("  Sample R-Bench image keys:", list(rbench_by_image.keys())[:5])
        print("  Our image IDs:", list(images.keys())[:5])


In [ ]:
# ============================================================
# Cell 4 — Stage 1: Caption Generation (Qwen3-VL-8B via Together.ai)
# ============================================================
# Qwen3-VL-8B generates captions with accurate object identification.
# RelCheck then focuses on fixing relational errors, not entity errors.

CAPTION_PROMPT = "Describe this image in 2-3 sentences. Focus on the objects present, the people (if any), and how they relate to each other spatially and through actions."


def encode_image_b64(image):
    buf = BytesIO()
    image.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


captions = {}
for img_id, img in images.items():
    b64 = encode_image_b64(img)
    try:
        resp = client.chat.completions.create(
            model=CAPTION_MODEL,
            messages=[{"role": "user", "content": [
                {"type": "text", "text": CAPTION_PROMPT},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
            ]}],
            temperature=0.0, max_tokens=200,
        )
        caption = resp.choices[0].message.content.strip()
        # Remove any thinking tags if Qwen outputs them
        caption = re.sub(r'<think>.*?</think>', '', caption, flags=re.DOTALL).strip()
    except Exception as e:
        caption = f"Caption generation failed: {e}"
        print(f"  [{img_id}] FAILED: {e}")

    captions[img_id] = caption
    print(f"[{img_id}] {caption[:150]}...")
    print()
    time.sleep(0.5)

print(f"Generated {len(captions)} captions.")


In [ ]:
# ============================================================
# Cell 5 — Stage 2: Visual Knowledge Base Construction
# ============================================================
# Three layers of evidence:
#   HARD  — GroundingDINO: objects, counts, bboxes (deterministic)
#   GEOM  — Bbox geometry: spatial relationships (deterministic)
#   SOFT  — Maverick VLM: actions, attributes, missing objects (visual)

DETECTION_THRESHOLD = 0.25

# Broad object categories to detect beyond what's in the caption
BROAD_CATEGORIES = [
    "person", "man", "woman", "child", "boy", "girl",
    "dog", "cat", "bird", "horse", "animal",
    "car", "bicycle", "motorcycle", "bus", "truck",
    "chair", "table", "bench", "couch", "bed",
    "food", "plate", "bowl", "cup", "bottle", "glass",
    "bag", "umbrella", "phone", "book", "sign",
]


def _clean_gdino_label(label):
    """Clean noisy GroundingDINO labels like 'bowl a bowl' or 'a red double-decker bus bus'."""
    label = label.strip().lower()
    # Split into words
    words = label.split()
    if len(words) <= 2:
        # Strip leading article
        if words[0] in ('a', 'an', 'the') and len(words) > 1:
            return ' '.join(words[1:])
        return label
    # Check for repeated subsequences — take the shortest meaningful label
    # e.g., "bowl a bowl" → "bowl", "person woman girl" → keep as-is (multi-label)
    # Strategy: find the shortest substring that appears in the original queries
    # For now: remove duplicate words and strip articles
    seen = []
    for w in words:
        if w in ('a', 'an', 'the'):
            continue
        if w not in seen:
            seen.append(w)
    return ' '.join(seen) if seen else label


def detect_objects(image, text_queries, threshold=DETECTION_THRESHOLD):
    """Run GroundingDINO. Returns list of (label, score, bbox_norm)."""
    text = ". ".join(text_queries) + "."
    inputs = gdino_processor(images=image, text=text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = gdino_model(**inputs)
    results = gdino_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold=threshold, text_threshold=threshold,
        target_sizes=[image.size[::-1]],
    )[0]
    detections = []
    W, H = image.size
    lkey = "text_labels" if "text_labels" in results else "labels"
    for score, label, box in zip(results["scores"], results[lkey], results["boxes"]):
        x1, y1, x2, y2 = box.tolist()
        # Clean noisy GroundingDINO labels (e.g., "bowl a bowl" → "bowl")
        label = _clean_gdino_label(label)
        detections.append((label, score.item(), [x1/W, y1/H, x2/W, y2/H]))
    return detections


def extract_nouns_from_caption(caption):
    """Extract noun phrases from caption for targeted detection."""
    import spacy
    nlp = spacy.load("en_core_web_sm")
    doc = nlp(caption)
    nouns = set()
    for chunk in doc.noun_chunks:
        # Get the head noun (last word usually)
        nouns.add(chunk.root.text.lower())
        # Also add full chunk for compound nouns
        nouns.add(chunk.text.lower().strip())
    return list(nouns)


def compute_spatial_facts(detections):
    """Compute pairwise spatial relationships from bboxes."""
    facts = []
    # Group detections by label for counting
    counts = Counter(label for label, _, _ in detections)

    for label, count in counts.items():
        if count > 1:
            facts.append(f"There are {count} instances of '{label}'")
        else:
            facts.append(f"There is 1 '{label}'")

    # Pairwise spatial relations for distinct object types
    unique_labels = list(counts.keys())
    for i, l1 in enumerate(unique_labels):
        det1 = [(s, b) for l, s, b in detections if l == l1]
        for j, l2 in enumerate(unique_labels):
            if i >= j:
                continue
            det2 = [(s, b) for l, s, b in detections if l == l2]
            # Use highest confidence detection for each
            _, b1 = max(det1, key=lambda x: x[0])
            _, b2 = max(det2, key=lambda x: x[0])

            c1x, c1y = (b1[0]+b1[2])/2, (b1[1]+b1[3])/2
            c2x, c2y = (b2[0]+b2[2])/2, (b2[1]+b2[3])/2

            # Containment check
            contained = (b1[0] >= b2[0]-0.05 and b1[2] <= b2[2]+0.05 and
                         b1[1] >= b2[1]-0.05 and b1[3] <= b2[3]+0.05)
            if contained:
                facts.append(f"'{l1}' is on/inside '{l2}' (spatially contained)")
                continue

            # Relative position
            dx, dy = c1x - c2x, c1y - c2y
            if abs(dx) > abs(dy):
                pos = "to the left of" if dx < 0 else "to the right of"
            else:
                pos = "above" if dy < 0 else "below"
            facts.append(f"'{l1}' is {pos} '{l2}'")

    return facts


def encode_image_b64(image):
    buf = BytesIO()
    image.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


MAVERICK_KB_PROMPT = """Describe the RELATIONSHIPS between objects in this image.

An object detector found these objects:
{detection_list}

For each pair of detected objects that interact, describe their relationship:
- SPATIAL: Where are they relative to each other? (on, next to, behind, inside, etc.)
- ACTIONS: What is each person/animal doing? What are they interacting with?
- ATTRIBUTES: Relevant attributes tied to relationships (wearing X, holding Y, made of Z)

RULES:
- Only describe what you can clearly see — no guesses
- Focus on relationships between the DETECTED objects listed above
- Use short, factual statements (e.g., "The boy is sitting on the bench", "The dog is next to the woman")
- Do NOT list objects that aren't interacting with anything

Format as a numbered list of relationship facts."""


# ── Build KB for each image ──
knowledge_bases = {}

for img_id, img in images.items():
    print(f"\n{'='*60}")
    print(f"[{img_id}] Building Visual Knowledge Base...")
    kb = {"hard_facts": [], "spatial_facts": [], "visual_description": ""}

    # ── 2a: GroundingDINO detection ──
    # Extract nouns from caption + add broad categories
    caption_nouns = extract_nouns_from_caption(captions[img_id])
    all_queries = list(set(caption_nouns + BROAD_CATEGORIES))

    detections = detect_objects(img, all_queries)
    # Deduplicate overlapping detections (keep highest confidence per label)
    seen = {}
    for label, score, bbox in sorted(detections, key=lambda x: -x[1]):
        # Check if we already have this label with overlapping bbox
        key = label.lower().strip()
        if key not in seen:
            seen[key] = [(score, bbox)]
        else:
            # Check IoU with existing detections of same label
            is_duplicate = False
            for _, existing_bbox in seen[key]:
                ix1 = max(bbox[0], existing_bbox[0])
                iy1 = max(bbox[1], existing_bbox[1])
                ix2 = min(bbox[2], existing_bbox[2])
                iy2 = min(bbox[3], existing_bbox[3])
                inter = max(0, ix2-ix1) * max(0, iy2-iy1)
                a1 = (bbox[2]-bbox[0])*(bbox[3]-bbox[1])
                a2 = (existing_bbox[2]-existing_bbox[0])*(existing_bbox[3]-existing_bbox[1])
                iou = inter / (a1 + a2 - inter + 1e-8)
                if iou > 0.5:
                    is_duplicate = True
                    break
            if not is_duplicate:
                seen[key].append((score, bbox))

    clean_detections = []
    for label, entries in seen.items():
        for score, bbox in entries:
            clean_detections.append((label, score, bbox))

    kb["detections"] = clean_detections

    # Format detection list for Maverick
    det_counts = Counter(l for l, _, _ in clean_detections)
    det_list_str = ""
    for label, count in det_counts.most_common():
        bboxes = [(s, b) for l, s, b in clean_detections if l == label]
        bbox_strs = [f"bbox={[round(x,2) for x in b]} conf={s:.2f}" for s, b in bboxes]
        det_list_str += f"- {label} ({count}x): {', '.join(bbox_strs)}\n"

    kb["hard_facts"].append(f"Detected {len(clean_detections)} objects total:")
    for label, count in det_counts.most_common():
        kb["hard_facts"].append(f"  {count}x {label}")

    print(f"  Detections: {len(clean_detections)} objects ({dict(det_counts)})")

    # ── 2b: Spatial geometry ──
    spatial_facts = compute_spatial_facts(clean_detections)
    kb["spatial_facts"] = spatial_facts
    print(f"  Spatial facts: {len(spatial_facts)}")

    # ── 2c: Maverick VLM descriptions ──
    b64 = encode_image_b64(img)
    prompt = MAVERICK_KB_PROMPT.format(detection_list=det_list_str)
    try:
        resp = client.chat.completions.create(
            model=VLM_MODEL,
            messages=[{"role": "user", "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
            ]}],
            temperature=0.0, max_tokens=500,
        )
        visual_desc = resp.choices[0].message.content.strip()
    except Exception as e:
        visual_desc = f"VLM description failed: {e}"
        print(f"  Maverick failed: {e}")

    kb["visual_description"] = visual_desc
    print(f"  Maverick description: {len(visual_desc)} chars")

    knowledge_bases[img_id] = kb
    time.sleep(0.5)

print(f"\n{'='*60}")
print(f"Built knowledge bases for {len(knowledge_bases)} images.")


In [ ]:
# ============================================================
# Cell 6 — Stage 3: Caption vs KB Comparison
# ============================================================
# Llama extracts triples and checks against KB.
# Programmatic post-check overrides Llama with hard evidence.
# Confidence scoring: only high-confidence contradictions get corrected.

COMPARE_PROMPT = """You are a strict caption fact-checker. Compare a caption against a Visual Knowledge Base (KB).
The KB was built by an object detector and a vision model looking at the actual image.

Caption: "{caption}"

=== VISUAL KNOWLEDGE BASE ===

DETECTED OBJECTS (from object detector — highly reliable):
{hard_facts}

SPATIAL FACTS (from bounding box geometry):
{spatial_facts}

VISUAL DESCRIPTION (from vision model):
{visual_description}

=== INSTRUCTIONS ===

Step 1: Extract ALL factual claims from the caption as (subject, relation, object) triples.
Examples:
  "a woman is putting on a hat" → (woman, putting on, hat)
  "people sitting on a table" → (people, sitting on, table)
  "a man eating a cupcake" → (man, eating, cupcake)
  "sitting in a bathroom" → (person, sitting in, bathroom)

Step 2: Check each triple against the KB using these STRICT rules:

  SUPPORTED — ONLY if KB explicitly confirms this claim with specific evidence
  CONTRADICTED — KB evidence directly disagrees with this claim. You must have CLEAR evidence.
  UNVERIFIABLE — KB has no info about this claim (THIS IS THE DEFAULT)

For CONTRADICTED verdicts, also rate your confidence:
  HIGH — multiple KB sources agree (e.g., detector AND VLM both disagree with caption)
  LOW — only one KB source disagrees, or the disagreement is subtle (e.g., "next to" vs "near")

When in doubt, mark UNVERIFIABLE. Do NOT use your own knowledge — only the KB.
You MUST extract at least one triple from any non-empty caption.

Output valid JSON only:
{{
  "triples": [
    {{"subject": "...", "relation": "...", "object": "...", "verdict": "SUPPORTED|CONTRADICTED|UNVERIFIABLE", "kb_evidence": "brief KB evidence", "confidence": "HIGH|LOW"}}
  ]
}}"""


# ── Programmatic post-check ──

ENTITY_SYNONYMS = {
    "person": ["person", "man", "woman", "boy", "girl", "child", "people", "worker"],
    "man": ["person", "man", "boy"],
    "woman": ["person", "woman", "girl"],
    "boy": ["person", "boy", "child", "man"],
    "girl": ["person", "girl", "child", "woman"],
    "child": ["person", "child", "boy", "girl"],
    "people": ["person", "man", "woman", "boy", "girl", "child"],
    "dog": ["dog", "animal"],
    "cat": ["cat", "animal"],
    "car": ["car", "vehicle"],
    "bus": ["bus", "vehicle"],
    "truck": ["truck", "vehicle"],
}

LOCATION_OBJECTS = {
    "bathroom": ["sink", "toilet", "mirror", "bathtub", "shower"],
    "kitchen": ["stove", "oven", "refrigerator", "sink", "microwave", "counter"],
    "bedroom": ["bed", "pillow", "blanket", "mattress"],
    "office": ["desk", "computer", "monitor", "keyboard", "chair"],
    "street": ["car", "bus", "road", "traffic", "sidewalk", "motorcycle"],
    "park": ["tree", "grass", "bench"],
    "beach": ["sand", "water", "ocean", "wave", "umbrella"],
    "restaurant": ["table", "chair", "plate", "food", "menu"],
}


def post_check_verdicts(triples, kb):
    """Override Llama verdicts using hard KB evidence. These are always HIGH confidence."""
    detected_labels = set()
    for fact in kb["hard_facts"]:
        fact_lower = fact.lower()
        for word in fact_lower.split():
            word = word.strip("'\"(),")
            if word and not word.isdigit() and word not in ("there", "is", "are", "of", "detected", "objects", "total", "instances"):
                detected_labels.add(word)

    if "detections" in kb:
        for label, score, bbox in kb["detections"]:
            for word in label.lower().split():
                detected_labels.add(word.strip())

    for triple in triples:
        subj = triple["subject"].lower().strip()
        obj = triple["object"].lower().strip()
        rel = triple["relation"].lower().strip()
        old_verdict = triple["verdict"]

        # Entity existence check
        for caption_word in subj.split() + obj.split():
            caption_word = caption_word.strip()
            if caption_word in ENTITY_SYNONYMS:
                synonyms = ENTITY_SYNONYMS[caption_word]
                found = any(syn in detected_labels for syn in synonyms)
                if not found:
                    triple["verdict"] = "CONTRADICTED"
                    triple["confidence"] = "HIGH"  # Detector is definitive
                    triple["kb_evidence"] = f"Detector found 0 {caption_word} (no {'/'.join(synonyms)} detected)"
                    break

        # Location consistency check
        for location, required_objects in LOCATION_OBJECTS.items():
            if location in subj or location in obj or location in rel:
                has_evidence = any(req in detected_labels for req in required_objects)
                if not has_evidence and triple["verdict"] == "SUPPORTED":
                    triple["verdict"] = "UNVERIFIABLE"
                    triple["confidence"] = "HIGH"
                    triple["kb_evidence"] = f"No {location}-related objects detected"

        if triple["verdict"] != old_verdict:
            print(f"    POST-CHECK: {old_verdict} → {triple['verdict']} "
                  f"({triple['subject']}, {triple['relation']}, {triple['object']})")

    return triples


def format_kb_text(kb):
    hard = "\n".join(f"- {f}" for f in kb["hard_facts"])
    spatial = "\n".join(f"- {f}" for f in kb["spatial_facts"]) if kb["spatial_facts"] else "- No spatial facts computed"
    visual = kb["visual_description"]
    return hard, spatial, visual


comparisons = {}

for img_id in images:
    caption = captions[img_id]
    kb = knowledge_bases[img_id]
    hard, spatial, visual = format_kb_text(kb)

    prompt = COMPARE_PROMPT.format(
        caption=caption, hard_facts=hard,
        spatial_facts=spatial, visual_description=visual,
    )

    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0, max_tokens=1500,
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r'^```json\s*', '', raw)
        raw = re.sub(r'```\s*$', '', raw)
        if raw.count('{') > raw.count('}'):
            raw += ']}'
        result = json.loads(raw)
        if "triples" not in result:
            result = {"triples": []}
    except Exception as e:
        print(f"  [{img_id}] Comparison failed: {e}")
        print(f"  Raw response: {raw[:200] if 'raw' in dir() else 'N/A'}")
        result = {"triples": []}

    # Ensure all triples have confidence field
    for t in result["triples"]:
        if "confidence" not in t:
            t["confidence"] = "LOW"  # Default if Llama didn't provide it

    # Programmatic post-check
    result["triples"] = post_check_verdicts(result["triples"], kb)

    comparisons[img_id] = result

    # Display
    triples = result.get("triples", [])
    n_sup = sum(1 for t in triples if t.get("verdict") == "SUPPORTED")
    n_con = sum(1 for t in triples if t.get("verdict") == "CONTRADICTED")
    n_unv = sum(1 for t in triples if t.get("verdict") == "UNVERIFIABLE")
    n_high = sum(1 for t in triples if t.get("verdict") == "CONTRADICTED" and t.get("confidence") == "HIGH")

    print(f"\n[{img_id}] {len(triples)} triples: {n_sup} supported, {n_con} contradicted ({n_high} high-conf), {n_unv} unverifiable")
    for t in triples:
        marker = "✓" if t["verdict"] == "SUPPORTED" else "✗" if t["verdict"] == "CONTRADICTED" else "?"
        conf = f" [{t.get('confidence', '?')}]" if t["verdict"] == "CONTRADICTED" else ""
        print(f"  {marker} ({t['subject']}, {t['relation']}, {t['object']}) — {t['verdict']}{conf}")
        if t["verdict"] == "CONTRADICTED":
            print(f"      KB says: {t.get('kb_evidence', '?')}")

    time.sleep(0.3)

# Summary
all_triples = [t for r in comparisons.values() for t in r.get("triples", [])]
all_contra = [t for t in all_triples if t["verdict"] == "CONTRADICTED"]
high_conf = [t for t in all_contra if t.get("confidence") == "HIGH"]
print(f"\n{'='*60}")
print(f"Total triples: {len(all_triples)}")
print(f"  SUPPORTED:    {sum(1 for t in all_triples if t['verdict']=='SUPPORTED')}")
print(f"  CONTRADICTED: {len(all_contra)} ({len(high_conf)} high-confidence)")
print(f"  UNVERIFIABLE: {sum(1 for t in all_triples if t['verdict']=='UNVERIFIABLE')}")
print(f"\nOnly HIGH-confidence contradictions will be corrected.")


In [ ]:
# ============================================================
# Cell 7 — Stage 4: High-Precision Caption Correction
# ============================================================
# Only fixes HIGH-confidence contradictions.
# After correction, verifies no information was lost.

CORRECTION_PROMPT = """Fix ONLY the numbered errors below in this caption. Keep EVERY other word exactly the same.

Caption: "{caption}"

Errors to fix:
{contradictions}

RULES:
- For each error, replace ONLY the wrong words with the correction shown
- Do NOT remove any other part of the caption
- Do NOT rephrase, shorten, or restructure any sentence
- Do NOT add new information
- Keep all punctuation, adjectives, and details that aren't listed as errors
- If an error says an entity doesn't exist, remove ONLY the minimal clause about that entity

Output ONLY the fixed caption:"""

VERIFY_PROMPT = """Compare these two captions. Does the second caption preserve all the key factual claims from the first, except for the specific changes listed?

Original: "{original}"
Corrected: "{corrected}"
Intended changes: {changes}

Check:
1. Are all details from the original preserved (except the intended changes)?
2. Does the corrected version introduce any NEW claims not in the original?

Answer with ONLY "PASS" if the correction is safe, or "FAIL: [reason]" if information was lost or new errors were added."""


def levenshtein(s1, s2):
    if len(s1) < len(s2): return levenshtein(s2, s1)
    if len(s2) == 0: return len(s1)
    prev = range(len(s2)+1)
    for c1 in s1:
        curr = [prev[0]+1]
        for j, c2 in enumerate(s2):
            curr.append(min(curr[-1]+1, prev[j+1]+1, prev[j]+(c1!=c2)))
        prev = curr
    return prev[-1]


final_results = {}

for img_id in images:
    caption = captions[img_id]
    comp = comparisons[img_id]
    triples = comp.get("triples", [])

    # ONLY correct HIGH-confidence contradictions
    high_conf_contradicted = [
        t for t in triples
        if t.get("verdict") == "CONTRADICTED" and t.get("confidence") == "HIGH"
    ]

    if not high_conf_contradicted:
        n_low = sum(1 for t in triples if t.get("verdict") == "CONTRADICTED" and t.get("confidence") != "HIGH")
        if n_low > 0:
            print(f"  [{img_id[:12]}] Skipping {n_low} low-confidence contradictions")
        final_results[img_id] = {
            "caption": caption, "corrected": caption,
            "triples": len(triples),
            "supported": sum(1 for t in triples if t["verdict"] == "SUPPORTED"),
            "contradicted": sum(1 for t in triples if t["verdict"] == "CONTRADICTED"),
            "unverifiable": sum(1 for t in triples if t["verdict"] == "UNVERIFIABLE"),
            "high_conf_contradicted": 0,
            "edit_rate": 0.0, "comparison": comp,
        }
        continue

    # Format contradictions as find→replace
    contra_str = ""
    changes_list = []
    for i, t in enumerate(high_conf_contradicted, 1):
        wrong_phrase = f"{t['subject']} {t['relation']} {t['object']}"
        kb_says = t.get('kb_evidence', 'not confirmed')
        contra_str += f"{i}. WRONG: \"{wrong_phrase}\"\n   CORRECTION: {kb_says}\n"
        changes_list.append(f"{wrong_phrase} → {kb_says}")

    prompt = CORRECTION_PROMPT.format(caption=caption, contradictions=contra_str)

    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0, max_tokens=400,
        )
        corrected = resp.choices[0].message.content.strip()
        corrected = re.sub(r'^(Corrected|Fixed|Here is[^:]*)[:\s]+', '',
                           corrected, flags=re.IGNORECASE).strip().strip('"').strip("'")
    except Exception as e:
        corrected = caption
        print(f"  [{img_id}] Correction failed: {e}")

    edit_rate = levenshtein(caption, corrected) / max(len(caption), 1)

    # ── Safety gate: reject if edit rate too high ──
    if edit_rate > 0.40:
        print(f"  [{img_id[:12]}] Edit rate {edit_rate:.0%} > 40% — too aggressive, keeping original")
        corrected = caption
        edit_rate = 0.0
    elif corrected != caption:
        # ── Verification: check correction didn't lose information ──
        try:
            verify_resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": VERIFY_PROMPT.format(
                    original=caption, corrected=corrected,
                    changes="; ".join(changes_list)
                )}],
                temperature=0.0, max_tokens=100,
            )
            verdict = verify_resp.choices[0].message.content.strip()
            if verdict.upper().startswith("FAIL"):
                print(f"  [{img_id[:12]}] Verification FAILED: {verdict}")
                print(f"    Keeping original caption")
                corrected = caption
                edit_rate = 0.0
            else:
                print(f"  [{img_id[:12]}] Verification PASSED — {len(high_conf_contradicted)} fixes, edit rate {edit_rate:.0%}")
        except Exception as e:
            # If verification fails, keep correction (benefit of the doubt)
            print(f"  [{img_id[:12]}] Verification call failed: {e} — keeping correction")

    final_results[img_id] = {
        "caption": caption, "corrected": corrected,
        "triples": len(triples),
        "supported": sum(1 for t in triples if t["verdict"] == "SUPPORTED"),
        "contradicted": sum(1 for t in triples if t["verdict"] == "CONTRADICTED"),
        "unverifiable": sum(1 for t in triples if t["verdict"] == "UNVERIFIABLE"),
        "high_conf_contradicted": len(high_conf_contradicted),
        "edit_rate": edit_rate, "comparison": comp,
    }

n_corrected = sum(1 for r in final_results.values() if r["corrected"] != r["caption"])
print(f"\nCorrections: {n_corrected}/{len(final_results)} images modified")
print(f"(Only high-confidence contradictions corrected, all verified)")


In [ ]:
# ============================================================
# Cell 8 — Display Results
# ============================================================
print("=" * 70)
print("RELCHECK V8 — KB-BASED ARCHITECTURE RESULTS")
print("=" * 70)

for img_id, r in final_results.items():
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: image with detections
    ax = axes[0]
    img = images[img_id]
    ax.imshow(img)
    kb = knowledge_bases[img_id]
    W, H = img.size
    colors = plt.cm.Set2(np.linspace(0, 1, max(len(kb["detections"]), 1)))
    for idx, (label, score, bbox) in enumerate(kb["detections"][:15]):
        x1, y1, x2, y2 = bbox[0]*W, bbox[1]*H, bbox[2]*W, bbox[3]*H
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                  edgecolor=colors[idx % len(colors)], facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1-3, f"{label} ({score:.2f})", fontsize=7,
                color=colors[idx % len(colors)], fontweight="bold", backgroundcolor="black")
    ax.set_title(f"[{img_id[:12]}] Detections", fontsize=10)
    ax.axis("off")

    # Right: caption comparison
    ax2 = axes[1]
    ax2.axis("off")

    orig = textwrap.fill(f"ORIGINAL: {r['caption']}", width=70)
    corr = textwrap.fill(f"CORRECTED: {r['corrected']}", width=70)
    stats = (f"Triples: {r['triples']} | Supported: {r['supported']} | "
             f"Contradicted: {r['contradicted']} | Unverifiable: {r['unverifiable']}")
    edit = f"Edit rate: {r['edit_rate']:.1%}"
    missing = ""  # Removed: not tracking missing objects (focus on relational corrections)

    # Show contradictions
    contra_lines = []
    for t in r["comparison"].get("triples", []):
        if t["verdict"] == "CONTRADICTED":
            contra_lines.append(f"  ✗ ({t['subject']}, {t['relation']}, {t['object']})")
            contra_lines.append(f"    → KB: {t.get('kb_evidence', '?')[:80]}")

    text = f"{orig}\n\n{corr}\n\n{stats}\n{edit}"
    if missing:
        text += f"\n{missing}"
    if contra_lines:
        text += f"\n\nContradictions:\n" + "\n".join(contra_lines)

    ax2.text(0, 1, text, transform=ax2.transAxes, fontsize=8, family="monospace",
             verticalalignment="top",
             bbox=dict(boxstyle="round,pad=0.5", facecolor="lightyellow", alpha=0.8))

    plt.tight_layout()
    plt.show()
    print()

# ── Summary Stats ──
N = len(final_results)
n_corrected = sum(1 for r in final_results.values() if r["corrected"] != r["caption"])
tot_t = sum(r["triples"] for r in final_results.values())
tot_s = sum(r["supported"] for r in final_results.values())
tot_c = sum(r["contradicted"] for r in final_results.values())
tot_u = sum(r["unverifiable"] for r in final_results.values())
corr_edits = [r["edit_rate"] for r in final_results.values() if r["corrected"] != r["caption"]]
avg_edit = sum(corr_edits) / max(len(corr_edits), 1)

print("=" * 70)
print(f"Images: {N} | Corrected: {n_corrected}/{N}")
print(f"Triples: {tot_t} ({tot_t/max(N,1):.1f}/img) | S:{tot_s} C:{tot_c} U:{tot_u}")
print(f"Avg edit rate (corrected): {avg_edit:.1%}")
print("=" * 70)


In [ ]:
# ============================================================
# Cell 9 — R-POPE Evaluation (LLM-Judge)
# ============================================================
# THE VIABILITY METRIC: does RelCheck improve caption accuracy
# on R-Bench ground-truth relational questions?
#
# For each image with R-Bench questions:
#   1. Llama (text-only) answers based on ORIGINAL caption → compare to GT
#   2. Llama (text-only) answers based on CORRECTED caption → compare to GT
#   3. If corrected accuracy > original accuracy → RelCheck works

if rbench_data is None or len(rbench_data) == 0:
    print("⚠️ R-Bench data not available — cannot run R-POPE evaluation.")
    print("Check Cell 3 output for download errors.")
else:
    RPOPE_PROMPT = """Based ONLY on this caption, answer the question with 'yes' or 'no'.
Do not use any external knowledge. Only use information stated in the caption.

Caption: "{caption}"
Question: {question}

Answer (yes or no):"""

    def rpope_judge(caption, question):
        """Ask Llama to answer a question based solely on the caption."""
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content":
                    RPOPE_PROMPT.format(caption=caption, question=question)}],
                temperature=0.0, max_tokens=5,
            )
            answer = resp.choices[0].message.content.strip().lower()
            if "yes" in answer and "no" not in answer:
                return "yes"
            elif "no" in answer:
                return "no"
            return "unknown"
        except Exception as e:
            print(f"    Judge error: {e}")
            return "unknown"

    # ── Build R-Bench lookup ──
    rbench_by_image = defaultdict(list)
    for entry in rbench_data:
        rbench_by_image[entry['image']].append(entry)

    # ── Match our test images to R-Bench ──
    img_to_questions = {}
    for our_id in images:
        for rb_key in rbench_by_image:
            rb_clean = rb_key.replace('.jpg', '').replace('.jpeg', '').replace('.png', '')
            if our_id == rb_clean or our_id in rb_key or rb_key in our_id:
                img_to_questions[our_id] = rbench_by_image[rb_key]
                break

    matched_images = [iid for iid in images if iid in img_to_questions]
    print(f"Matched {len(matched_images)}/{len(images)} images with R-Bench questions.")

    if not matched_images:
        print("\n⚠️ No image ID matches found.")
        print("R-Bench image keys:", list(rbench_by_image.keys())[:5])
        print("Our image IDs:", list(images.keys())[:5])
        print("\nThe images may not overlap with R-Bench. Try using R-Bench images directly.")
    else:
        orig_correct, corr_correct, total_q = 0, 0, 0
        per_image_results = []

        for img_id in matched_images:
            questions = img_to_questions[img_id]
            orig_cap = final_results[img_id]["caption"]
            corr_cap = final_results[img_id]["corrected"]
            img_orig_c, img_corr_c, img_total = 0, 0, 0

            print(f"\n[{img_id[:12]}] {len(questions)} R-Bench questions")
            print(f"  Original:  {orig_cap[:80]}...")
            print(f"  Corrected: {corr_cap[:80]}...")

            for entry in questions[:10]:  # Up to 10 questions per image
                question = entry['question']
                gt = entry['answer']

                orig_ans = rpope_judge(orig_cap, question)
                corr_ans = rpope_judge(corr_cap, question)

                if orig_ans == gt:
                    orig_correct += 1
                    img_orig_c += 1
                if corr_ans == gt:
                    corr_correct += 1
                    img_corr_c += 1
                total_q += 1
                img_total += 1

                # Show changes
                if orig_ans != corr_ans:
                    change = "✅ IMPROVED" if corr_ans == gt else "❌ REGRESSED"
                    print(f"    Q: {question[:60]}")
                    print(f"      GT={gt} | Orig={orig_ans} | Corr={corr_ans} → {change}")

            if img_total > 0:
                per_image_results.append({
                    "img_id": img_id,
                    "orig_acc": img_orig_c / img_total,
                    "corr_acc": img_corr_c / img_total,
                    "n_questions": img_total,
                    "was_corrected": final_results[img_id]["corrected"] != final_results[img_id]["caption"],
                })

            time.sleep(0.3)

        # ── Summary ──
        print(f"\n{'='*60}")
        print(f"R-POPE (LLM-Judge) RESULTS — {total_q} questions across {len(matched_images)} images")
        print(f"{'='*60}")

        if total_q > 0:
            orig_acc = orig_correct / total_q
            corr_acc = corr_correct / total_q
            delta = corr_correct - orig_correct

            print(f"  Original caption accuracy:  {orig_correct}/{total_q} ({orig_acc:.1%})")
            print(f"  Corrected caption accuracy: {corr_correct}/{total_q} ({corr_acc:.1%})")
            print(f"  Delta: {'+' if delta >= 0 else ''}{delta} questions ({delta/total_q:+.1%})")

            if delta > 0:
                print(f"\n  ✅ RelCheck IMPROVES relational accuracy by {delta/total_q:+.1%}")
            elif delta == 0:
                print(f"\n  ➖ No change in accuracy")
            else:
                print(f"\n  ❌ RelCheck DECREASED accuracy by {delta/total_q:.1%}")

            # Per-image breakdown
            print(f"\n  Per-image breakdown:")
            for r in per_image_results:
                marker = "📝" if r["was_corrected"] else "—"
                d = r["corr_acc"] - r["orig_acc"]
                print(f"    {marker} [{r['img_id'][:12]}] Orig={r['orig_acc']:.0%} → Corr={r['corr_acc']:.0%} "
                      f"({'+' if d >= 0 else ''}{d:.0%}) [{r['n_questions']}q]")

        print(f"{'='*60}")

    # ── Save results ──
    save_data = {}
    for img_id, r in final_results.items():
        save_data[img_id] = {
            "caption": r["caption"],
            "corrected": r["corrected"],
            "edit_rate": r["edit_rate"],
            "triples": r["triples"],
            "supported": r["supported"],
            "contradicted": r["contradicted"],
            "unverifiable": r.get("unverifiable", 0),
            "kb_hard_facts": knowledge_bases[img_id]["hard_facts"],
            "kb_spatial_facts": knowledge_bases[img_id]["spatial_facts"],
            "kb_visual_description": knowledge_bases[img_id]["visual_description"],
            "comparison": r["comparison"],
        }
    with open("viability_v8_results.json", "w") as f:
        json.dump(save_data, f, indent=2, default=str)
    print("\nResults saved to viability_v8_results.json")


In [ ]:
# ============================================================
# Cell 10 — Full Analysis: Every Image, Every Detail
# ============================================================
# Shows complete captions, KB facts, all triples + verdicts,
# every R-POPE question with original/corrected answers vs GT.

print("=" * 80)
print("FULL ANALYSIS — RelCheck v8 Viability Test")
print("=" * 80)

for idx, img_id in enumerate(images):
    r = final_results[img_id]
    kb = knowledge_bases[img_id]
    comp = comparisons[img_id]
    was_corrected = r["corrected"] != r["caption"]

    print(f"\n{'━' * 80}")
    print(f"IMAGE {idx+1}: {img_id}")
    print(f"{'━' * 80}")

    # ── Full captions ──
    print(f"\n📝 ORIGINAL CAPTION:")
    print(f"   {r['caption']}")
    if was_corrected:
        print(f"\n✏️  CORRECTED CAPTION:")
        print(f"   {r['corrected']}")
        print(f"   Edit rate: {r['edit_rate']:.1%}")
    else:
        print(f"\n   (No correction applied)")

    # ── KB facts ──
    print(f"\n🔍 KNOWLEDGE BASE:")
    print(f"   Detected objects:")
    for fact in kb["hard_facts"]:
        print(f"     • {fact}")
    if kb["spatial_facts"]:
        print(f"   Spatial facts:")
        for fact in kb["spatial_facts"]:
            print(f"     • {fact}")
    print(f"   VLM description:")
    for line in kb["visual_description"].split("\n"):
        line = line.strip()
        if line:
            print(f"     {line}")

    # ── All triples + verdicts ──
    triples = comp.get("triples", [])
    print(f"\n⚖️  TRIPLES ({len(triples)} total):")
    for t in triples:
        if t["verdict"] == "SUPPORTED":
            icon = "✅"
        elif t["verdict"] == "CONTRADICTED":
            conf = t.get("confidence", "?")
            icon = f"❌[{conf}]"
        else:
            icon = "❓"
        print(f"   {icon} ({t['subject']}, {t['relation']}, {t['object']}) — {t['verdict']}")
        if t.get("kb_evidence"):
            print(f"        Evidence: {t['kb_evidence']}")

    # ── R-POPE questions ──
    if img_id in img_to_questions:
        questions = img_to_questions[img_id]
        print(f"\n📊 R-POPE QUESTIONS ({len(questions)}):")
        for entry in questions[:10]:
            question = entry["question"]
            gt = entry["answer"]

            orig_ans = rpope_judge(r["caption"], question)
            corr_ans = rpope_judge(r["corrected"], question)

            orig_mark = "✅" if orig_ans == gt else "❌"
            corr_mark = "✅" if corr_ans == gt else "❌"

            # Highlight changes
            if orig_ans != corr_ans:
                if corr_ans == gt:
                    delta = " ⬆️ IMPROVED"
                else:
                    delta = " ⬇️ REGRESSED"
            else:
                delta = ""

            print(f"\n   Q: {question}")
            print(f"   GT: {gt}")
            print(f"   Original answer:  {orig_ans} {orig_mark}")
            print(f"   Corrected answer: {corr_ans} {corr_mark}{delta}")
    else:
        print(f"\n   (No R-Bench questions for this image)")

print(f"\n{'━' * 80}")
print("END OF ANALYSIS")
print(f"{'━' * 80}")
